In [13]:
config = {
    "vocab_size": 32000,
    "seq_len": 128,
    "n_layers": 4,
    "n_heads": 4,
    "d_model": 256,
    "dropout": 0.1,
}

In [14]:
import torch
import torch.nn as nn
import math

class Head(nn.Module):
    def __init__(self, d_model, head_size):
        super().__init__()

        self.key = nn.Linear(d_model, head_size, bias=False)
        self.query = nn.Linear(d_model, head_size, bias=False)
        self.value = nn.Linear(d_model, head_size, bias=False)

    def forward(self, x):
        B, T, C = x.shape

        k = self.key(x)
        q = self.query(x)

        scores = q @ k.transpose(-2, -1)
        scores = scores / math.sqrt(k.size(-1))

        weights = torch.softmax(scores, dim=-1)

        v = self.value(x)

        out = weights @ v

        return out

In [15]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()

        head_size = d_model // n_heads

        self.heads = nn.ModuleList([
            Head(d_model, head_size)
            for _ in range(n_heads)
        ])

        self.proj = nn.Linear(d_model, d_model)

    def forward(self, x):
        out = torch.cat(
            [h(x) for h in self.heads],
            dim=-1
        )

        return self.proj(out)

In [16]:
class FeedForward(nn.Module):
    def __init__(self, d_model):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model)
        )

    def forward(self, x):
        return self.net(x)

In [17]:
class Block(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()

        self.attn = MultiHeadAttention(d_model, n_heads)
        self.ff = FeedForward(d_model)

        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

In [18]:
class GPT(nn.Module):
    def __init__(
        self,
        vocab_size,
        seq_len,
        d_model,
        n_heads,
        n_layers
    ):
        super().__init__()

        self.token_embedding = nn.Embedding(
            vocab_size,
            d_model
        )

        self.position_embedding = nn.Embedding(
            seq_len,
            d_model
        )

        self.blocks = nn.Sequential(
            *[
                Block(d_model, n_heads)
                for _ in range(n_layers)
            ]
        )

        self.ln_f = nn.LayerNorm(d_model)

        self.head = nn.Linear(
            d_model,
            vocab_size
        )

    def forward(self, idx):
        B, T = idx.shape

        tok_emb = self.token_embedding(idx)

        pos = torch.arange(T, device=idx.device)

        pos_emb = self.position_embedding(pos)

        x = tok_emb + pos_emb

        x = self.blocks(x)

        x = self.ln_f(x)

        logits = self.head(x)

        return logits

In [19]:
model = GPT(
    vocab_size=32000,
    seq_len=128,
    d_model=256,
    n_heads=4,
    n_layers=4
)

params = sum(
    p.numel()
    for p in model.parameters()
)

print(params)

19605248


In [1]:
import torch

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Torch version: 2.11.0+cu128
CUDA available: True
CUDA version: 12.8
GPU count: 1
GPU: NVIDIA GeForce RTX 3060 Laptop GPU


In [27]:
import sys
import torch

print("Python:", sys.executable)
print("Torch:", torch.__file__)
print("Version:", torch.__version__)


Python: c:\Users\dande\Pytorch\.conda\python.exe
Torch: c:\Users\dande\Pytorch\.conda\Lib\site-packages\torch\__init__.py
Version: 2.12.1+cpu


In [25]:
import sys

!{sys.executable} -m pip uninstall -y torch torchvision torchaudio
!{sys.executable} -m pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128

Found existing installation: torch 2.11.0+cu128
Uninstalling torch-2.11.0+cu128:
  Successfully uninstalled torch-2.11.0+cu128
Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128
Looking in indexes: https://download.pytorch.org/whl/cu128
  Using cached torch-2.11.0%2Bcu128-cp311-cp311-win_amd64.whl.metadata (29 kB)
  Using cached torchvision-0.26.0%2Bcu128-cp311-cp311-win_amd64.whl.metadata (5.6 kB)
  Using cached torchaudio-2.11.0%2Bcu128-cp311-cp311-win_amd64.whl.metadata (7.0 kB)
Using cached torch-2.11.0%2Bcu128-cp311-cp311-win_amd64.whl (2753.1 MB)
Using cached torchvision-0.26.0%2Bcu128-cp311-cp311-win_amd64.whl (9.3 MB)
Using cached torchaudio-2.11.0%2Bcu128-cp311-cp311-win_amd64.whl (1.7 MB)

   ---------------------------------------- 0/3 [torcha

In [1]:
import torch
import time

a = torch.randn((12000, 12000), device="cuda")
b = torch.randn((12000, 12000), device="cuda")

torch.cuda.synchronize()

start = time.time()

for _ in range(20):
    c = a @ b

torch.cuda.synchronize()

print("Time:", time.time() - start)

Time: 10.153429985046387


In [2]:
print(torch.cuda.get_device_name(0))
print(torch.cuda.memory_summary())

NVIDIA GeForce RTX 3060 Laptop GPU
|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |   1658 MiB |   2208 MiB |  12108 MiB |  10450 MiB |
|       from large pool |   1658 MiB |   2208 MiB |  12108 MiB |  10450 MiB |
|       from small pool |      0 MiB |      0 MiB |      0 MiB |      0 MiB |
|---------------------------------------------------------------------------|
| Active memory         |   1658 MiB |   2208 MiB |  12108 MiB |  10450 MiB |
|       from large pool |   1

In [3]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

tensor_size = (10000, 10000)  
a = torch.randn(tensor_size, device=device)  
b = torch.randn(tensor_size, device=device)  

c = a + b  

print("Result shape (moved to CPU for printing):", c.cpu().shape)

print("Current GPU memory usage:")
print(f"Allocated: {torch.cuda.memory_allocated(device) / (1024 ** 2):.2f} MB")
print(f"Cached: {torch.cuda.memory_reserved(device) / (1024 ** 2):.2f} MB")

Using device: cuda
Result shape (moved to CPU for printing): torch.Size([10000, 10000])
Current GPU memory usage:
Allocated: 1152.53 MB
Cached: 2220.00 MB
